# Racenet.com.au Horse Racing Scraper

All-in-one notebook that:
1. Installs dependencies
2. Scrapes one week of Australian race results from racenet.com.au
3. Saves raw JSON files per race
4. Builds a flat training dataset (JSONL + CSV)
5. Previews the collected data

**Run cells top-to-bottom in order.**

## 1. Install Dependencies

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'playwright', 'beautifulsoup4'], check=True)
subprocess.run([sys.executable, '-m', 'playwright', 'install', 'chromium'], check=True)
print('Done.')

## 2. Imports & Configuration

In [ ]:
import asyncio
import csv
import json
import logging
import re
import statistics
from datetime import date, datetime, timedelta
from pathlib import Path
from typing import Optional

from bs4 import BeautifulSoup
from playwright.async_api import async_playwright, Browser, BrowserContext, Page

# ── Scraper settings ──────────────────────────────────────────────────────────
BASE_URL    = 'https://www.racenet.com.au'
RESULTS_URL = f'{BASE_URL}/results/horse-racing'

HEADLESS      = True    # set False to watch the browser
DAYS_TO_SCRAPE = 7      # how many past days to collect
OUTPUT_DIR    = Path('data')   # raw JSON output folder
DATASET_DIR   = Path('dataset') # flat training data output

RACE_DELAY    = 1.5     # seconds between race pages
MEETING_DELAY = 2.0     # seconds between meeting pages
PAGE_TIMEOUT  = 30_000  # ms

# Known non-Australian venue keywords — meetings matching these are skipped
OVERSEAS_KEYWORDS = [
    'hong-kong', 'singapore', 'newmarket', 'ascot-uk', 'cheltenham',
    'goodwood', 'ireland', 'france', 'usa', 'japan', 'south-africa',
    'uae', 'dubai',
]

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('racenet')
print('Config ready.')

## 3. Utility Functions

In [ ]:
def clean(t) -> str:
    return ' '.join(str(t).split()) if t else ''

def to_float(s) -> Optional[float]:
    try:
        return float(re.sub(r'[^0-9.\-]', '', str(s)))
    except (ValueError, TypeError):
        return None

def to_int(s) -> Optional[int]:
    try:
        return int(re.sub(r'[^0-9\-]', '', str(s)))
    except (ValueError, TypeError):
        return None

def parse_date(s) -> Optional[date]:
    for fmt in ('%Y-%m-%d', '%d/%m/%Y', '%d/%m/%y', '%d-%m-%Y'):
        try:
            return datetime.strptime(str(s).strip(), fmt).date()
        except ValueError:
            continue
    return None

def safe_mean(vals: list) -> Optional[float]:
    clean_vals = [v for v in vals if v is not None]
    return statistics.mean(clean_vals) if clean_vals else None

def finish_pos_int(pos) -> Optional[int]:
    if pos is None:
        return None
    s = str(pos).strip().upper()
    if s in ('DNF', 'SCR', 'DQ', 'PU', 'UR', 'NS', 'NP', 'DISQ', 'W/D'):
        return 999
    m = re.match(r'^(\d+)', s)
    return int(m.group(1)) if m else None

def is_australian(meeting_url: str) -> bool:
    lower = meeting_url.lower()
    return not any(kw in lower for kw in OVERSEAS_KEYWORDS)

print('Utilities ready.')

## 4. Scraping Functions

In [ ]:
async def click_tab(page: Page, label: str) -> bool:
    """Click a tab/button whose visible text contains *label*."""
    for sel in [
        f"button:has-text('{label}')",
        f"[role='tab']:has-text('{label}')",
        f"a:has-text('{label}')",
    ]:
        try:
            btn = page.locator(sel).first
            if await btn.count() > 0:
                await btn.click()
                await page.wait_for_load_state('networkidle', timeout=10_000)
                return True
        except Exception:
            pass
    return False


async def get_meetings_for_date(page: Page, target: date) -> list[dict]:
    """Return all Australian meeting dicts for *target* date."""
    date_str = target.strftime('%Y%m%d')
    url = f'{RESULTS_URL}?date={target.isoformat()}'
    log.info('Loading results index for %s', target)

    await page.goto(url, timeout=PAGE_TIMEOUT)
    await page.wait_for_load_state('networkidle', timeout=PAGE_TIMEOUT)
    await asyncio.sleep(1.5)

    soup = BeautifulSoup(await page.content(), 'html.parser')
    pattern = re.compile(r'/results/horse-racing/([^/]+)-(' + date_str + r')/?$')

    meetings, seen = [], set()
    for a in soup.find_all('a', href=True):
        href = a['href']
        m = pattern.search(href)
        if m and href not in seen:
            seen.add(href)
            full_url = BASE_URL + href if href.startswith('/') else href
            if is_australian(full_url):
                meetings.append({
                    'venue': m.group(1).replace('-', ' ').title(),
                    'venue_slug': m.group(1),
                    'url': full_url,
                    'date': target.isoformat(),
                })

    log.info('  Found %d AU meetings', len(meetings))
    return meetings


async def get_races_for_meeting(page: Page, meeting: dict) -> list[dict]:
    """Return all race dicts for a meeting."""
    await page.goto(meeting['url'], timeout=PAGE_TIMEOUT)
    await page.wait_for_load_state('networkidle', timeout=PAGE_TIMEOUT)
    await asyncio.sleep(1.0)

    soup = BeautifulSoup(await page.content(), 'html.parser')
    pattern = re.compile(r'/results/horse-racing/[^/]+-\d{8}/(.+-race-(\d+))/?$')

    races, seen = [], set()
    for a in soup.find_all('a', href=True):
        href = a['href']
        m = pattern.search(href)
        if m and href not in seen:
            seen.add(href)
            races.append({
                'race_number': int(m.group(2)),
                'race_slug': m.group(1),
                'url': BASE_URL + href if href.startswith('/') else href,
            })

    races.sort(key=lambda r: r['race_number'])
    log.info('  %s: %d races', meeting['venue'], len(races))
    return races


def _parse_results_table(soup: BeautifulSoup) -> tuple[dict, list]:
    """Extract race meta and runner list from the Results tab HTML."""
    meta = {}

    # Race header — try common class patterns
    for sel in ['[class*="raceHeader"]', '[class*="race-header"]',
                '[class*="RaceDetail"]', 'h1', 'h2']:
        el = soup.select_one(sel)
        if el:
            meta['race_header_raw'] = clean(el.get_text())
            break

    raw = meta.get('race_header_raw', '')

    m = re.search(r'(\d{3,5})\s*m\b', raw, re.I)
    if m:
        meta['distance_m'] = int(m.group(1))

    m = re.search(r'\b(Firm|Good|Soft|Heavy|Synthetic|Slow)\s*(\d)?', raw, re.I)
    if m:
        meta['track_condition'] = clean(m.group(0))

    m = re.search(r'\$\s*([\d,]+)', raw)
    if m:
        meta['prize_money'] = int(m.group(1).replace(',', ''))

    runners = []
    table = soup.find('table')
    if table:
        headers = [clean(th.get_text()).lower() for th in table.find_all('th')]
        for tr in table.find_all('tr')[1:]:
            cells = [clean(td.get_text()) for td in tr.find_all('td')]
            if not cells:
                continue
            row = dict(zip(headers, cells)) if headers else {}
            runners.append({
                'finish_position': row.get('pos') or row.get('position') or (cells[0] if cells else None),
                'barrier':         row.get('barrier') or row.get('draw') or None,
                'horse_name':      row.get('horse') or row.get('name') or (cells[2] if len(cells) > 2 else None),
                'jockey':          row.get('jockey') or row.get('rider') or None,
                'trainer':         row.get('trainer') or None,
                'weight':          row.get('weight') or row.get('wt') or None,
                'margin':          row.get('margin') or row.get('margins') or None,
                'win_odds':        row.get('odds') or row.get('win') or row.get('sp') or None,
            })

    return meta, runners


def _parse_form_table(soup: BeautifulSoup, runners: list[dict]) -> list[dict]:
    """Attach past-run history to each runner from the Form tab HTML."""
    known = {r['horse_name'].upper(): r['horse_name']
             for r in runners if r.get('horse_name')}

    form_by_horse: dict[str, list] = {}

    for table in soup.find_all('table'):
        horse_name = None
        prev = table.find_previous(['h3', 'h4', 'h5', 'strong', 'b', 'span'])
        if prev:
            candidate = clean(prev.get_text()).upper()
            for upper, orig in known.items():
                if upper in candidate or candidate in upper:
                    horse_name = orig
                    break
        if not horse_name:
            continue

        headers = [clean(th.get_text()).lower() for th in table.find_all('th')]
        past_runs = []
        for tr in table.find_all('tr')[1:]:
            cells = [clean(td.get_text()) for td in tr.find_all('td')]
            if not cells:
                continue
            row = dict(zip(headers, cells)) if headers else {}
            past_runs.append({
                'date':            row.get('date') or row.get('race date') or (cells[0] if cells else None),
                'venue':           row.get('track') or row.get('venue') or row.get('course') or None,
                'distance_m':      to_int(row.get('dist') or row.get('distance') or ''),
                'track_condition': row.get('cond') or row.get('condition') or row.get('going') or None,
                'race_class':      row.get('class') or row.get('race class') or None,
                'barrier':         to_int(row.get('barrier') or row.get('bar') or ''),
                'weight':          row.get('weight') or row.get('wt') or None,
                'jockey':          row.get('jockey') or row.get('rider') or None,
                'finish_position': row.get('pos') or row.get('position') or None,
                'margin':          row.get('margin') or None,
                'time':            row.get('time') or row.get('sectional') or None,
                'win_odds':        to_float(row.get('odds') or row.get('sp') or ''),
                'rating':          row.get('rating') or row.get('rtg') or None,
            })

        if past_runs:
            form_by_horse[horse_name] = past_runs

    for runner in runners:
        runner['form_history'] = form_by_horse.get(runner.get('horse_name'), [])

    return runners


async def scrape_race(page: Page, race: dict, meeting: dict) -> Optional[dict]:
    """Load one race page, scrape Results + Form tabs, return structured dict."""
    url = race['url']
    log.info('    Race %d: %s', race['race_number'], url)

    try:
        await page.goto(url, timeout=PAGE_TIMEOUT)
        await page.wait_for_load_state('networkidle', timeout=PAGE_TIMEOUT)
        await asyncio.sleep(1.0)
    except Exception as e:
        log.error('Failed to load %s: %s', url, e)
        return None

    soup = BeautifulSoup(await page.content(), 'html.parser')
    meta, runners = _parse_results_table(soup)

    meta.update({
        'url':         url,
        'race_number': race['race_number'],
        'race_slug':   race.get('race_slug', ''),
        'venue':       meeting['venue'],
        'venue_slug':  meeting['venue_slug'],
        'date':        meeting['date'],
        'scraped_at':  datetime.utcnow().isoformat() + 'Z',
    })

    if await click_tab(page, 'Form'):
        await asyncio.sleep(1.5)
        form_soup = BeautifulSoup(await page.content(), 'html.parser')
        runners = _parse_form_table(form_soup, runners)
    else:
        log.debug('    Form tab not found for %s', url)
        for r in runners:
            r['form_history'] = []

    return {'meta': meta, 'runners': runners}


print('Scraping functions ready.')

## 5. Save / Index Helpers

In [ ]:
def save_race(race_data: dict, out_dir: Path) -> Path:
    d = race_data['meta']
    folder = out_dir / d['date'] / d['venue_slug']
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / f"race_{d['race_number']:02d}.json"
    path.write_text(json.dumps(race_data, indent=2, ensure_ascii=False), encoding='utf-8')
    return path


def update_index(index_path: Path, race_data: dict) -> None:
    index = json.loads(index_path.read_text()) if index_path.exists() else []
    d = race_data['meta']
    entry = {
        'date':         d['date'],
        'venue':        d['venue'],
        'venue_slug':   d['venue_slug'],
        'race_number':  d['race_number'],
        'race_slug':    d.get('race_slug', ''),
        'url':          d['url'],
        'runners_count': len(race_data['runners']),
    }
    index = [e for e in index if not (
        e['date'] == entry['date'] and
        e['venue_slug'] == entry['venue_slug'] and
        e['race_number'] == entry['race_number']
    )]
    index.append(entry)
    index.sort(key=lambda e: (e['date'], e['venue_slug'], e['race_number']))
    index_path.write_text(json.dumps(index, indent=2, ensure_ascii=False), encoding='utf-8')


print('Save helpers ready.')

## 6. Main Crawler

In [ ]:
async def crawl(
    days: int = DAYS_TO_SCRAPE,
    out_dir: Path = OUTPUT_DIR,
    start_date: Optional[date] = None,
    resume: bool = True,
) -> list[dict]:
    """
    Crawl `days` days backwards from `start_date` (default: yesterday).
    Returns list of all scraped race dicts.
    """
    if start_date is None:
        start_date = date.today() - timedelta(days=1)

    dates = [start_date - timedelta(days=i) for i in range(days)]
    out_dir.mkdir(parents=True, exist_ok=True)
    index_path = out_dir / 'index.json'

    scraped_keys: set = set()
    if resume and index_path.exists():
        existing = json.loads(index_path.read_text())
        scraped_keys = {(e['date'], e['venue_slug'], e['race_number']) for e in existing}
        log.info('Resume: %d races already indexed', len(scraped_keys))

    all_races = []

    async with async_playwright() as pw:
        browser: Browser = await pw.chromium.launch(
            headless=HEADLESS,
            args=['--no-sandbox', '--disable-blink-features=AutomationControlled'],
        )
        ctx: BrowserContext = await browser.new_context(
            user_agent=(
                'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                'AppleWebKit/537.36 (KHTML, like Gecko) '
                'Chrome/124.0.0.0 Safari/537.36'
            ),
            viewport={'width': 1280, 'height': 800},
        )
        page: Page = await ctx.new_page()
        page.set_default_timeout(PAGE_TIMEOUT)

        for target_date in dates:
            log.info('=== %s ===', target_date)
            try:
                meetings = await get_meetings_for_date(page, target_date)
            except Exception as e:
                log.error('Meetings fetch failed for %s: %s', target_date, e)
                continue

            for meeting in meetings:
                await asyncio.sleep(MEETING_DELAY)
                try:
                    races = await get_races_for_meeting(page, meeting)
                except Exception as e:
                    log.error('Races fetch failed for %s: %s', meeting['url'], e)
                    continue

                for race in races:
                    key = (meeting['date'], meeting['venue_slug'], race['race_number'])
                    if resume and key in scraped_keys:
                        log.debug('Skipping %s', key)
                        continue

                    await asyncio.sleep(RACE_DELAY)
                    try:
                        race_data = await scrape_race(page, race, meeting)
                    except Exception as e:
                        log.error('Race scrape failed %s: %s', race['url'], e)
                        continue

                    if race_data:
                        save_race(race_data, out_dir)
                        update_index(index_path, race_data)
                        scraped_keys.add(key)
                        all_races.append(race_data)
                        log.info('    Saved race %d @ %s', race['race_number'], meeting['venue'])

        await browser.close()

    log.info('Crawl complete. %d races scraped.', len(all_races))
    return all_races


print('Crawler ready.')

## 7. Run the Scraper

This scrapes the last **7 days** of Australian race results.
Expect it to take 20–40 minutes depending on how many meetings ran.
Progress is logged below the cell. Re-running will skip already-scraped races.

In [ ]:
scraped_races = await crawl(
    days=DAYS_TO_SCRAPE,
    out_dir=OUTPUT_DIR,
    resume=True,
)

print(f'\nTotal races collected this run: {len(scraped_races)}')

## 8. Build Flat Training Dataset

In [ ]:
def extract_form_features(form_history: list, race_dist: Optional[int],
                           track_cond: Optional[str]) -> dict:
    positions  = [finish_pos_int(r.get('finish_position')) for r in form_history]
    positions  = [p for p in positions if p is not None and p < 999]
    odds_list  = [to_float(r.get('win_odds')) for r in form_history]
    odds_list  = [o for o in odds_list if o is not None]
    run_dates  = [parse_date(r.get('date')) for r in form_history]

    feats = {}

    for n in (3, 5, 10):
        feats[f'avg_finish_pos_last{n}'] = safe_mean(positions[:n])

    total = min(len(positions), 10)
    feats['win_rate_last10']  = sum(1 for p in positions[:10] if p == 1) / total if total else None
    feats['top3_rate_last10'] = sum(1 for p in positions[:10] if p <= 3) / total if total else None

    valid_dates = [d for d in run_dates if d is not None]
    feats['days_since_last_run'] = (date.today() - valid_dates[0]).days if valid_dates else None

    feats['avg_win_odds_last5'] = safe_mean(odds_list[:5])

    if race_dist is not None:
        dp = [finish_pos_int(r.get('finish_position')) for r in form_history
              if abs((to_int(r.get('distance_m')) or 0) - race_dist) <= 100]
        dp = [p for p in dp if p is not None and p < 999]
        feats['avg_finish_pos_similar_dist'] = safe_mean(dp[:5])
        feats['runs_at_similar_dist']        = len(dp)
    else:
        feats['avg_finish_pos_similar_dist'] = None
        feats['runs_at_similar_dist']        = None

    if track_cond:
        cond_key = track_cond.lower().split()[0]
        cp = [finish_pos_int(r.get('finish_position')) for r in form_history
              if (r.get('track_condition') or '').lower().startswith(cond_key)]
        cp = [p for p in cp if p is not None and p < 999]
        feats['avg_finish_pos_same_condition'] = safe_mean(cp[:5])
        feats['runs_on_same_condition']        = len(cp)
    else:
        feats['avg_finish_pos_same_condition'] = None
        feats['runs_on_same_condition']        = None

    feats['career_starts'] = len(form_history)
    feats['career_wins']   = sum(1 for r in form_history
                                 if finish_pos_int(r.get('finish_position')) == 1)
    return feats


def flatten_race(race_data: dict) -> list[dict]:
    meta    = race_data.get('meta', {})
    runners = race_data.get('runners', [])
    dist    = to_int(meta.get('distance_m') or '')
    cond    = meta.get('track_condition')
    rows    = []
    for r in runners:
        fp = finish_pos_int(r.get('finish_position'))
        rows.append({
            'race_date':     meta.get('date'),
            'venue':         meta.get('venue'),
            'race_number':   meta.get('race_number'),
            'distance_m':    dist,
            'track_condition': cond,
            'prize_money':   to_int(meta.get('prize_money') or ''),
            'horse_name':    r.get('horse_name'),
            'jockey':        r.get('jockey'),
            'trainer':       r.get('trainer'),
            'barrier':       to_int(r.get('barrier') or ''),
            'weight_kg':     to_float(re.sub(r'[^0-9.]', '', r.get('weight') or '')),
            'win_odds':      to_float(r.get('win_odds')),
            'finish_position': fp,
            'is_winner':     1 if fp == 1 else 0,
            'is_top3':       1 if (fp is not None and fp <= 3) else 0,
            'margin':        r.get('margin'),
            **extract_form_features(r.get('form_history', []), dist, cond),
        })
    return rows


def build_dataset(data_dir: Path = OUTPUT_DIR, out_dir: Path = DATASET_DIR) -> list[dict]:
    out_dir.mkdir(parents=True, exist_ok=True)
    race_files = sorted(data_dir.rglob('race_*.json'))
    print(f'Race files found: {len(race_files)}')

    all_rows = []
    for path in race_files:
        try:
            all_rows.extend(flatten_race(json.loads(path.read_text())))
        except Exception as e:
            print(f'  ERROR {path}: {e}')

    jsonl_path = out_dir / 'races.jsonl'
    jsonl_path.write_text(
        '\n'.join(json.dumps(r, ensure_ascii=False) for r in all_rows),
        encoding='utf-8'
    )

    if all_rows:
        csv_path = out_dir / 'dataset.csv'
        with open(csv_path, 'w', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=list(all_rows[0].keys()))
            w.writeheader()
            w.writerows(all_rows)
        print(f'Written: {csv_path}')

    print(f'Written: {jsonl_path}')
    print(f'Total runner records: {len(all_rows)}')
    return all_rows


all_rows = build_dataset()
print('Dataset build complete.')

## 9. Data Inspection

In [ ]:
# ── Summary stats ─────────────────────────────────────────────────────────────
index_path = OUTPUT_DIR / 'index.json'
if index_path.exists():
    index = json.loads(index_path.read_text())
    dates   = sorted({e['date'] for e in index})
    venues  = sorted({e['venue'] for e in index})
    print(f'Dates scraped  : {len(dates)}  ({dates[0]} → {dates[-1]})')
    print(f'Unique venues  : {len(venues)}')
    print(f'Total races    : {len(index)}')
    print(f'Total runners  : {len(all_rows)}')
    print()
    print('Venues:', ', '.join(venues))
else:
    print('No index found — scraper may not have collected data yet.')

In [ ]:
# ── Races per day ─────────────────────────────────────────────────────────────
if index_path.exists():
    from collections import Counter
    per_day = Counter(e['date'] for e in index)
    print('Races per day:')
    for d in sorted(per_day):
        print(f'  {d}: {per_day[d]} races')

In [ ]:
# ── Sample raw race JSON ───────────────────────────────────────────────────────
race_files = sorted(OUTPUT_DIR.rglob('race_*.json'))
if race_files:
    sample = json.loads(race_files[0].read_text())
    print('=== META ===')
    print(json.dumps(sample['meta'], indent=2))
    print()
    print(f'=== RUNNERS ({len(sample["runners"])} starters) ===')
    for r in sample['runners'][:3]:   # first 3 runners
        print(json.dumps({k: v for k, v in r.items() if k != 'form_history'}, indent=2))
        print(f'  form_history: {len(r.get("form_history", []))} past runs')
        print()
else:
    print('No race files found yet.')

In [ ]:
# ── Sample flat training rows ──────────────────────────────────────────────────
if all_rows:
    print('Column names:')
    print(list(all_rows[0].keys()))
    print()
    print('First 3 rows:')
    for row in all_rows[:3]:
        print(json.dumps(row, indent=2))
        print()

In [ ]:
# ── Feature completeness check ─────────────────────────────────────────────────
# Shows what % of rows have each field populated — useful to spot scraping gaps
if all_rows:
    fields = list(all_rows[0].keys())
    n = len(all_rows)
    print(f'Field completeness across {n} runner records:\n')
    for field in fields:
        filled = sum(1 for r in all_rows if r.get(field) is not None)
        pct = filled / n * 100
        bar = '█' * int(pct / 5) + '░' * (20 - int(pct / 5))
        print(f'  {field:<35s} {bar} {pct:5.1f}%')